# ML-10 — Content Action Playbook

This notebook turns our validated machine learning scoring model (**Lane 2: Refresh / Content Opportunity Scoring**) into an operational, human-reviewed Content Action Playbook and exports reproducible outputs to `work/outputs/` and `work/figures/`.

> Skill loaded: `skills/writing-honest-claims/SKILL.md` & `skills/flyrank/flyrank-data/SKILL.md`

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

# 1. Load dataset
data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# 2. Preprocessing & Feature Engineering
df["scroll_rate_filled"] = df["scroll_rate"].fillna(0)
df["engagement_rate_filled"] = df["engagement_rate"].fillna(0)
df["ai_traffic_pct_filled"] = df["ai_traffic_pct"].fillna(0)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["search_volume_filled"] = df["search_volume"].fillna(0)
df["competition_filled"] = df["competition"].fillna(0)
df["cpc_filled"] = df["cpc"].fillna(0)
df["word_count_filled"] = df["word_count"].fillna(df["word_count"].median())
df["stale_flag"] = (df["days_since_last_update"] >= 180).astype(int)
df["high_impression_flag"] = (df["impressions_90d"] >= 500).astype(int)
df["impressions_per_day"] = df["impressions_90d"] / (df["content_age_days"] + 1)

content_type_dummies = pd.get_dummies(df["content_type"], prefix="type", drop_first=True)
intent_dummies = pd.get_dummies(df["main_intent"].fillna("unknown"), prefix="intent", drop_first=True)

honest_numeric_cols = [
    "content_age_days", "days_since_last_update", "stale_flag",
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d",
    "engaged_sessions_90d", "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate_filled", "scroll_rate_filled", "ai_traffic_pct_filled",
    "search_volume_filled", "competition_filled", "cpc_filled",
    "word_count_filled", "has_keyword_data", "has_word_count",
    "high_impression_flag", "impressions_per_day"
]

X = pd.concat([df[honest_numeric_cols], content_type_dummies, intent_dummies], axis=1)
X = X.loc[:, ~X.columns.duplicated()]
y = df["is_declining_label"]

# 3. Train Random Forest Model
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X, y)
df["opportunity_score"] = rf.predict_proba(X)[:, 1]

# 4. Action & Reason Code Mapping
def assign_action_and_reason(row):
    score = row["opportunity_score"]
    stale = row["stale_flag"]
    high_imp = row["high_impression_flag"]
    word_cnt = row["word_count_filled"]
    ctr = row["ctr"]
    
    if score >= 0.6 and high_imp == 1 and stale == 1:
        return "HIGH_PRIORITY_REFRESH", "HIGH_VISIBILITY_STALE"
    elif score >= 0.6 and word_cnt < 1000:
        return "CONTENT_EXPANSION", "THIN_CONTENT_DECAY"
    elif score >= 0.5 and ctr < 0.5 and high_imp == 1:
        return "RE_OPTIMIZE_CTR", "LOW_CTR_HIGH_IMPRESSIONS"
    elif score >= 0.5:
        return "GENERAL_REFRESH", "TREND_DECLINE_RISK"
    else:
        return "MONITOR_ONLY", "STABLE_PERFORMANCE"

action_results = df.apply(assign_action_and_reason, axis=1)
df["recommended_action"] = [r[0] for r in action_results]
df["reason_code"] = [r[1] for r in action_results]

# Rank Queue by Opportunity Score
ranked_queue = df.sort_values("opportunity_score", ascending=False).reset_index(drop=True)

print("=== RANKED ACTION PLAYBOOK QUEUE (Top 5 Items) ===")
display_cols = ["content_id", "client_id", "opportunity_score", "recommended_action", "reason_code", "impressions_90d", "days_since_last_update"]
print(ranked_queue[display_cols].head(5).to_string(index=False))

print("\n=== RECOMMENDED ACTION DISTRIBUTION ===")
print(ranked_queue["recommended_action"].value_counts().to_string())

=== RANKED ACTION PLAYBOOK QUEUE (Top 5 Items) ===
          content_id         client_id  opportunity_score recommended_action              reason_code  impressions_90d  days_since_last_update
content_e74933316051 client_4ec9599fc2           0.761623    RE_OPTIMIZE_CTR LOW_CTR_HIGH_IMPRESSIONS             1674                       8
content_20c5d6a1c7b1 client_624b60c58c           0.755685    GENERAL_REFRESH       TREND_DECLINE_RISK              201                       8
content_6e1add102562 client_624b60c58c           0.751292    GENERAL_REFRESH       TREND_DECLINE_RISK              133                       8
content_9d35d489e74d client_624b60c58c           0.750342    GENERAL_REFRESH       TREND_DECLINE_RISK              169                       8
content_41ffdf98a18e client_624b60c58c           0.748589    GENERAL_REFRESH       TREND_DECLINE_RISK              247                       8

=== RECOMMENDED ACTION DISTRIBUTION ===
recommended_action
RE_OPTIMIZE_CTR          11304


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use & Boundaries

* **Intended Audience & Purpose:** Designed for editorial lead teams, content managers, and SEO strategists to prioritize manual content review and refresh schedules.
* **Decision-Support Scope:** The system outputs an ordered candidate queue of pages exhibiting observed traffic and engagement decay patterns.
* **Operational Limits:**
  * Does **not** predict Google algorithm updates or penalties.
  * Does **not** measure off-page signals (such as competitor backlink acquisition).
  * Does **not** evaluate brand tone, legal disclaimers, or factual accuracy.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Protocol & Strict No-Go Rules

#### Mandatory Human Review Checklist
Before executing any content refresh recommended by the playbook, human editors must:
1. **Audit Search Intent:** Verify whether user search intent for the target query shifted since publication.
2. **Verify Fact & Reference Accuracy:** Confirm all statistics, dates, product links, and citations are current.
3. **Evaluate Content Quality:** Ensure updates add genuine value rather than padding word count.

#### Strict No-Go Automation List
The following actions **must never be automated** by scripts or AI pipelines:
1. **No Automated URL Deletions or 301 Redirects:** Deleting URLs without human audit risks destroying external backlink equity and traffic.
2. **No Unchecked LLM Text Generation:** Auto-publishing generated text without human editorial review introduces hallucination risk.
3. **No Automated Edits to Compliance / Legal / Medical Content:** Pages involving legal terms, medical advice, or financial disclaimers require certified expert review.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Drift Monitoring & Retraining Triggers

To prevent model performance decay and stale recommendations, the playbook relies on four explicit monitoring triggers:
1. **Scheduled Retraining Window:** Re-fit feature scaling and model parameters every **90 days** as new GSC and GA4 trailing metrics accumulate.
2. **Performance Drift Trigger:** Re-evaluate scoring if top-50 Precision@50 drops below **0.50** on new client-holdout validation runs.
3. **Distribution Shift Trigger:** Re-calibrate if overall portfolio declining rate shifts by more than **±10 percentage points** from the baseline rate ($0.5421$).
4. **System Integration Drift:** Re-train immediately if GSC or GA4 metric definition schemas change.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure export directories exist relative to repo root / work folder
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent.parent

outputs_dir = repo_root / "work" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

figures_dir = repo_root / "work" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# 1. Export Ranked Action Playbook Queue CSV
export_queue_cols = [
    "content_id", "client_id", "opportunity_score", "recommended_action", "reason_code",
    "content_age_days", "days_since_last_update", "impressions_90d", "clicks_90d", "ctr", "avg_position"
]
queue_export_path = outputs_dir / "action_playbook_queue.csv"
ranked_queue[export_queue_cols].to_csv(queue_export_path, index=False)
print(f"Exported ranked queue CSV: {queue_export_path} ({len(ranked_queue):,} rows)")

# 2. Export Playbook Metrics Summary JSON
metrics_summary = {
    "total_pages_evaluated": len(df),
    "base_declining_rate": float(df["is_declining_label"].mean()),
    "action_distribution": df["recommended_action"].value_counts().to_dict(),
    "reason_code_distribution": df["reason_code"].value_counts().to_dict(),
    "top_opportunity_score": float(ranked_queue["opportunity_score"].max()),
    "median_opportunity_score": float(ranked_queue["opportunity_score"].median())
}
metrics_export_path = outputs_dir / "action_playbook_metrics.json"
with open(metrics_export_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Exported playbook metrics JSON: {metrics_export_path}")

# 3. Export Action Distribution Chart Figure
plt.figure(figsize=(8, 4.5))
action_counts = df["recommended_action"].value_counts()
bars = plt.barh(action_counts.index, action_counts.values, color="#1f77b4")
plt.title("Content Action Playbook Queue Distribution")
plt.xlabel("Number of Content Items")
plt.gca().invert_yaxis()
plt.tight_layout()

chart_export_path = figures_dir / "action_mix.png"
plt.savefig(chart_export_path, dpi=150)
plt.close()
print(f"Exported action distribution chart: {chart_export_path}")

Exported ranked queue CSV: /sessions/gallant-loving-wright/mnt/ML/FLY_Manthan/work/outputs/action_playbook_queue.csv (30,000 rows)
Exported playbook metrics JSON: /sessions/gallant-loving-wright/mnt/ML/FLY_Manthan/work/outputs/action_playbook_metrics.json
Exported action distribution chart: /sessions/gallant-loving-wright/mnt/ML/FLY_Manthan/work/figures/action_mix.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.